<a href="https://colab.research.google.com/github/SouravDasz/Deep-learning-projects-models-only-/blob/main/attention_mechanism.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset,DataLoader
import torch.nn as nn
import torch.optim as optim


In [2]:
df=pd.read_csv("/content/english_to_bangla (1).csv")

In [3]:
df.head()

,en,bn
0,a child in a pink dress is climbing up a set o...,একটি গোলাপী জামা পরা বাচ্চা মেয়ে একটি বাড়ির প্...
1,a girl going into a wooden building .,একটি মেয়ে শিশু একটি কাঠের বাড়িতে ঢুকছে
2,a little girl climbing into a wooden playhouse .,একটি বাচ্চা তার কাঠের খেলাঘরে উঠছে ।
3,a little girl climbing the stairs to her playh...,ছোট মেয়েটি তার খেলার ঘরের সিড়ি বেয়ে উঠছে
4,a little girl in a pink dress going into a woo...,গোলাপি জামা পড়া ছোট একটি মেয়ে একটি কাঠের তৈরি...


In [4]:
df["en"][100]

'two different breeds of brown and white dogs play on the beach .'

In [5]:
import re

In [6]:
def tokenizing(x):
  x=x.lower()
  x=re.sub("[^a-zA-Z]"," ",x)
  x=x.split()
  return x


In [7]:
df["en"]=df["en"].apply(tokenizing)

In [8]:
df["en"][5]

['a', 'black', 'dog', 'and', 'a', 'spotted', 'dog', 'are', 'fighting']

In [9]:
import re

def tokenizing_bn(x):
    x = x.lower()
    x = re.sub(r"[^\u0980-\u09FF]+", " ", x)
    x = x.split()
    return x

In [10]:
df["bn"]=df["bn"].apply(tokenizing_bn)

In [11]:
df

,en,bn
0,"[a, child, in, a, pink, dress, is, climbing, u...","[একটি, গোলাপী, জামা, পরা, বাচ্চা, মেয়ে, একটি, ..."
1,"[a, girl, going, into, a, wooden, building]","[একটি, মেয়ে, শিশু, একটি, কাঠের, বাড়িতে, ঢুকছে]"
2,"[a, little, girl, climbing, into, a, wooden, p...","[একটি, বাচ্চা, তার, কাঠের, খেলাঘরে, উঠছে]"
3,"[a, little, girl, climbing, the, stairs, to, h...","[ছোট, মেয়েটি, তার, খেলার, ঘরের, সিড়ি, বেয়ে, উঠছে]"
4,"[a, little, girl, in, a, pink, dress, going, i...","[গোলাপি, জামা, পড়া, ছোট, একটি, মেয়ে, একটি, কা..."
...,...,...
39060,"[a, man, in, a, pink, shirt, climbs, a, rock, ...","[গোলাপী, শার্টের, একটি, লোক, একটি, শিলার, উপরে..."
39061,"[a, man, is, rock, climbing, high, in, the, air]","[একজন, মানুষ, পাথরে, চড়ছে, অনেক, উপরে]"
39062,"[a, person, in, a, red, shirt, climbing, up, a...","[লাল, শার্টের, একজন, ব্যক্তি, সহায়তার, হাতলগু..."
39063,"[a, rock, climber, in, a, red, shirt]","[লাল, শার্টে, একটি, রক, আরোহী]"


In [12]:
df["bn"][1]

['একটি', 'মেয়ে', 'শিশু', 'একটি', 'কাঠের', 'বাড়িতে', 'ঢুকছে']

# Integer Encoding

In [13]:
from collections import Counter

In [14]:
en_vocab={
    "<oov>":0,
}
bn_vocab={
    "<oov>":0,
    "<sos>":1,
    "<eos>":2
}

In [15]:
en_counter=Counter()
bn_counter=Counter()

In [16]:
for sentence in df["en"]:
  en_counter.update(sentence)

In [17]:
en_counter

Counter({'a': 61112,
         'child': 1532,
         'in': 18406,
         'pink': 732,
         'dress': 347,
         'is': 9039,
         'climbing': 478,
         'up': 1273,
         'set': 109,
         'of': 6590,
         'stairs': 106,
         'an': 2393,
         'entry': 1,
         'way': 53,
         'girl': 3295,
         'going': 143,
         'into': 1045,
         'wooden': 272,
         'building': 504,
         'little': 1748,
         'playhouse': 6,
         'the': 17672,
         'to': 3113,
         'her': 1169,
         'cabin': 4,
         'black': 3659,
         'dog': 7596,
         'and': 8634,
         'spotted': 37,
         'are': 3412,
         'fighting': 123,
         'tri': 14,
         'colored': 216,
         'playing': 1896,
         'with': 7559,
         'each': 408,
         'other': 744,
         'on': 10433,
         'road': 389,
         'white': 3797,
         'brown': 2449,
         'spots': 28,
         'staring': 57,
         'at': 2869

In [18]:
for sentence in df["bn"]:
  bn_counter.update(sentence)

In [19]:
bn_counter

Counter({'একটি': 23972,
         'গোলাপী': 308,
         'জামা': 1368,
         'পরা': 2286,
         'বাচ্চা': 1684,
         'মেয়ে': 1453,
         'বাড়ির': 15,
         'প্রবেশ': 24,
         'পথের': 41,
         'সিঁড়ি': 29,
         'বেয়ে': 282,
         'উঠছে': 376,
         'শিশু': 1484,
         'কাঠের': 395,
         'বাড়িতে': 1,
         'ঢুকছে': 5,
         'তার': 2340,
         'খেলাঘরে': 5,
         'ছোট': 2245,
         'মেয়েটি': 460,
         'খেলার': 355,
         'ঘরের': 99,
         'সিড়ি': 11,
         'গোলাপি': 319,
         'পড়া': 672,
         'তৈরি': 158,
         'ঘরে': 70,
         'করছে': 4477,
         'কালো': 3225,
         'কুকুর': 7248,
         'এবং': 4414,
         'ছোপওয়ালা': 13,
         'ঝগড়া': 17,
         'তিন': 304,
         'রঙা': 8,
         'কুকুরের': 630,
         'সাথে': 1225,
         'রাস্তায়': 568,
         'খেলছে': 2338,
         'ও': 1869,
         'সাদা': 3254,
         'বাদামি': 767,
         'ছোপযুক্ত': 58,
         'রাস্তায়': 604,
 

In [20]:
for word in en_counter:
  if word not in en_vocab:
    en_vocab[word]=len(en_vocab)

In [21]:
en_vocab

{'<oov>': 0,
 'a': 1,
 'child': 2,
 'in': 3,
 'pink': 4,
 'dress': 5,
 'is': 6,
 'climbing': 7,
 'up': 8,
 'set': 9,
 'of': 10,
 'stairs': 11,
 'an': 12,
 'entry': 13,
 'way': 14,
 'girl': 15,
 'going': 16,
 'into': 17,
 'wooden': 18,
 'building': 19,
 'little': 20,
 'playhouse': 21,
 'the': 22,
 'to': 23,
 'her': 24,
 'cabin': 25,
 'black': 26,
 'dog': 27,
 'and': 28,
 'spotted': 29,
 'are': 30,
 'fighting': 31,
 'tri': 32,
 'colored': 33,
 'playing': 34,
 'with': 35,
 'each': 36,
 'other': 37,
 'on': 38,
 'road': 39,
 'white': 40,
 'brown': 41,
 'spots': 42,
 'staring': 43,
 'at': 44,
 'street': 45,
 'two': 46,
 'dogs': 47,
 'different': 48,
 'breeds': 49,
 'looking': 50,
 'pavement': 51,
 'moving': 52,
 'toward': 53,
 'covered': 54,
 'paint': 55,
 'sits': 56,
 'front': 57,
 'painted': 58,
 'rainbow': 59,
 'hands': 60,
 'bowl': 61,
 'sitting': 62,
 'large': 63,
 'small': 64,
 'grass': 65,
 'plays': 66,
 'fingerpaints': 67,
 'canvas': 68,
 'it': 69,
 'there': 70,
 'pigtails': 71,
 'pa

In [22]:
for word in bn_counter:
  if word not in bn_vocab:
    bn_vocab[word]=len(bn_vocab)

In [23]:
bn_vocab


{'<oov>': 0,
 '<sos>': 1,
 '<eos>': 2,
 'একটি': 3,
 'গোলাপী': 4,
 'জামা': 5,
 'পরা': 6,
 'বাচ্চা': 7,
 'মেয়ে': 8,
 'বাড়ির': 9,
 'প্রবেশ': 10,
 'পথের': 11,
 'সিঁড়ি': 12,
 'বেয়ে': 13,
 'উঠছে': 14,
 'শিশু': 15,
 'কাঠের': 16,
 'বাড়িতে': 17,
 'ঢুকছে': 18,
 'তার': 19,
 'খেলাঘরে': 20,
 'ছোট': 21,
 'মেয়েটি': 22,
 'খেলার': 23,
 'ঘরের': 24,
 'সিড়ি': 25,
 'গোলাপি': 26,
 'পড়া': 27,
 'তৈরি': 28,
 'ঘরে': 29,
 'করছে': 30,
 'কালো': 31,
 'কুকুর': 32,
 'এবং': 33,
 'ছোপওয়ালা': 34,
 'ঝগড়া': 35,
 'তিন': 36,
 'রঙা': 37,
 'কুকুরের': 38,
 'সাথে': 39,
 'রাস্তায়': 40,
 'খেলছে': 41,
 'ও': 42,
 'সাদা': 43,
 'বাদামি': 44,
 'ছোপযুক্ত': 45,
 'রাস্তায়': 46,
 'একে': 47,
 'অপরের': 48,
 'দিকে': 49,
 'তাকিয়ে': 50,
 'আছে': 51,
 'ভিন্ন': 52,
 'জাতের': 53,
 'দুটি': 54,
 'রাস্তার': 55,
 'পাশে': 56,
 'দুইটি': 57,
 'পরস্পরের': 58,
 'এগিয়ে': 59,
 'যাচ্ছ': 60,
 'রঙে': 61,
 'মাখা': 62,
 'হাতে': 63,
 'রঙের': 64,
 'বাটিতে': 65,
 'হাত': 66,
 'দিয়ে': 67,
 'রঙ': 68,
 'আঁকা': 69,
 'রংধনুর': 70,
 'সামনে': 71,
 'বসে': 72,
 'ছোটো': 73,

In [24]:
en_vocab

{'<oov>': 0,
 'a': 1,
 'child': 2,
 'in': 3,
 'pink': 4,
 'dress': 5,
 'is': 6,
 'climbing': 7,
 'up': 8,
 'set': 9,
 'of': 10,
 'stairs': 11,
 'an': 12,
 'entry': 13,
 'way': 14,
 'girl': 15,
 'going': 16,
 'into': 17,
 'wooden': 18,
 'building': 19,
 'little': 20,
 'playhouse': 21,
 'the': 22,
 'to': 23,
 'her': 24,
 'cabin': 25,
 'black': 26,
 'dog': 27,
 'and': 28,
 'spotted': 29,
 'are': 30,
 'fighting': 31,
 'tri': 32,
 'colored': 33,
 'playing': 34,
 'with': 35,
 'each': 36,
 'other': 37,
 'on': 38,
 'road': 39,
 'white': 40,
 'brown': 41,
 'spots': 42,
 'staring': 43,
 'at': 44,
 'street': 45,
 'two': 46,
 'dogs': 47,
 'different': 48,
 'breeds': 49,
 'looking': 50,
 'pavement': 51,
 'moving': 52,
 'toward': 53,
 'covered': 54,
 'paint': 55,
 'sits': 56,
 'front': 57,
 'painted': 58,
 'rainbow': 59,
 'hands': 60,
 'bowl': 61,
 'sitting': 62,
 'large': 63,
 'small': 64,
 'grass': 65,
 'plays': 66,
 'fingerpaints': 67,
 'canvas': 68,
 'it': 69,
 'there': 70,
 'pigtails': 71,
 'pa

In [25]:
def en_converter(x):
  return [en_vocab.get(word) for word in x]
def bn_converter(x):
  return [bn_vocab.get(word) for word in x]

In [26]:
df['en']=df["en"].apply(en_converter)
df["bn"]=df["bn"].apply(bn_converter)

In [27]:
df

,en,bn
0,"[1, 2, 3, 1, 4, 5, 6, 7, 8, 1, 9, 10, 11, 3, 1...","[3, 4, 5, 6, 7, 8, 3, 9, 10, 11, 12, 13, 14]"
1,"[1, 15, 16, 17, 1, 18, 19]","[3, 8, 15, 3, 16, 17, 18]"
2,"[1, 20, 15, 7, 17, 1, 18, 21]","[3, 7, 19, 16, 20, 14]"
3,"[1, 20, 15, 7, 22, 11, 23, 24, 21]","[21, 22, 19, 23, 24, 25, 13, 14]"
4,"[1, 20, 15, 3, 1, 4, 5, 16, 17, 1, 18, 25]","[26, 5, 27, 21, 3, 8, 3, 16, 28, 29, 10, 30]"
...,...,...
39060,"[1, 75, 3, 1, 4, 161, 112, 1, 164, 202]","[4, 2805, 3, 107, 3, 1066, 239, 5650]"
39061,"[1, 75, 6, 164, 7, 487, 3, 22, 227]","[92, 97, 702, 1325, 516, 239]"
39062,"[1, 187, 3, 1, 110, 161, 7, 8, 1, 164, 202, 54...","[140, 2805, 92, 103, 15657, 15658, 243, 3, 126..."
39063,"[1, 164, 288, 3, 1, 110, 161]","[140, 4049, 3, 1249, 288]"


In [28]:
df["en"][0]

[1, 2, 3, 1, 4, 5, 6, 7, 8, 1, 9, 10, 11, 3, 12, 13, 14]

# Train Test Spliting

In [29]:
x_train, x_test, y_train, y_test = train_test_split(
    df["en"], df["bn"], test_size=0.2, random_state=42
)

# Padding

In [30]:
max_len = max(len(seq) for seq in x_train)
print(max_len)
max_len=max(len(seq) for seq in x_test)
print(max_len)

33
36


In [31]:
en_max_len=36

In [32]:
max_len=max( len(seq) for seq in y_train)
print(max_len)
max_len=max(len(seq) for seq in y_test)
print(max_len)


32
29


In [33]:
bn_max_len=32

In [34]:
from torch.nn.utils.rnn import pad_sequence

In [35]:
def padding(seq, actual_max_len):
  if len(seq) < actual_max_len:
    return seq + [0] * (actual_max_len - len(seq))
  else :
    return seq

In [36]:
x_train = [padding(seq, en_max_len) for seq in x_train]
x_test  = [padding(seq, en_max_len) for seq in x_test]

In [37]:
x_train[1]

[1,
 75,
 35,
 1,
 168,
 2061,
 6,
 62,
 80,
 1,
 491,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0]

In [38]:
y_train = [padding(seq, bn_max_len) for seq in y_train]
y_test  = [padding(seq, bn_max_len) for seq in y_test]

In [39]:
y_train[1]

[228,
 6110,
 3,
 107,
 722,
 168,
 72,
 389,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0]

# Dataset and Dataloader

In [40]:
class MDataset(Dataset):
  def __init__(self,feature,labels):
    self.feature=feature
    self.labels=labels
  def __len__(self):
    return len(self.feature)
  def __getitem__(self, index):
    x=torch.tensor(self.feature[index], dtype=torch.long)
    y=torch.tensor(self.labels[index], dtype=torch.long)
    return x,y

In [41]:
data=MDataset(x_train,y_train)

In [42]:
data

In [43]:
len(data)

31252

In [44]:
data_loader=DataLoader(data,num_workers=0,batch_size=128)

In [45]:
len(data_loader)

245

# Model creation

In [46]:
class Encoder(nn.Module):
  def __init__(self):
    super().__init__()
    self.emb=nn.Embedding(len(en_vocab),embedding_dim=50)
    self.lstm=nn.LSTM(50,hidden_size=128,batch_first=True)
  def forward(self,x):
    x=self.emb(x)
    output,(hidden,cell)=self.lstm(x)
    return output,hidden,cell


In [47]:
class Attention(nn.Module):
  def __init__(self):
    super().__init__()
  def forward(self,hidden,encoder_output):
    hidden=hidden.permute(1,2,0)
    score=torch.bmm(encoder_output,hidden).squeeze(2)
    weight=torch.softmax(score,dim=1)
    return weight

In [48]:
class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(len(bn_vocab), embedding_dim=50)
        self.attention = Attention()
        self.lstm = nn.LSTM(50 + 128, hidden_size=128, batch_first=True)
        #                    ↑    ↑
        #                  emb  hidden from attention context
        self.fc = nn.Linear(128, len(bn_vocab))

    def forward(self, target, hidden, cell, encoder_output):
        # target: (batch,) one word at a time

        # Step 1 — embed the input word
        target = target.unsqueeze(1)                          # (batch, 1)
        embedded = self.emb(target)                           # (batch, 1, 50)

        # Step 2 — attention
        weight = self.attention(hidden, encoder_output)       # (batch, src_len)
        weight = weight.unsqueeze(1)                          # (batch, 1, src_len)

        # Step 3 — context vector
        context = torch.bmm(weight, encoder_output)           # (batch, 1, 128)

        # Step 4 — concat embedding + context
        lstm_input = torch.cat([embedded, context], dim=2)   # (batch, 1, 50+128)

        # Step 5 — lstm
        output, (hidden, cell) = self.lstm(lstm_input, (hidden, cell))
        #                                               ↑
        #                                   pass hidden and cell forward

        # Step 6 — predict next word
        pred = self.fc(output.squeeze(1))                     # (batch, fr_vocab)

        return pred, hidden, cell


In [49]:
import random

In [60]:
class Seq2Seq(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = Encoder()
        self.decoder = Decoder()

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        # src: (batch, src_len)
        # trg: (batch, trg_len)

        batch_size = src.shape[0]
        trg_len    = trg.shape[1]

        # store predictions
        predictions = torch.zeros(batch_size, trg_len, len(bn_vocab)).to(src.device)

        # Step 1 — run encoder once
        encoder_output, hidden, cell = self.encoder(src)

        # Step 2 — first decoder input is <SOS> token
        input = trg[:, 0]                                      # (batch,)

        # Step 3 — decode word by word
        for t in range(1, trg_len):

            pred, hidden, cell = self.decoder(
                input, hidden, cell, encoder_output
            )
            predictions[:, t, :] = pred                        # store prediction

            # teacher forcing
            teacher_force = random.random() < teacher_forcing_ratio
            if teacher_force:
                input = trg[:, t]                              # use real word
            else:
                input = pred.argmax(1)                         # use predicted word

        return predictions

In [51]:
device="cuda" if torch.cuda.is_available() else "cpu"

In [52]:
device

'cuda'

In [62]:
import random
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# ── hyperparameters ──────────────────────────────
EPOCHS              = 10
BATCH_SIZE          = 32
LEARNING_RATE       = 0.01
TEACHER_FORCE_RATIO = 0.5
CLIP                = 1.0          # gradient clipping
pad_idx             = en_vocab['<oov>']

# ── model, optimizer, loss ───────────────────────
model     = Seq2Seq().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)
#                                ↑
#                   ignore PAD tokens in loss calculation

In [57]:
def train(model, dataloader, optimizer, criterion, clip):
    model.train()                          # set model to train mode
    epoch_loss = 0

    for src, trg in dataloader:
        if isinstance(src, list):
            src = torch.stack(src)         # Stack list of tensors into a single tensor
        src = src.to(device)               # (batch, src_len)

        if isinstance(trg, list):
            trg = torch.stack(trg)         # Stack list of tensors into a single tensor
        trg = trg.to(device)               # (batch, trg_len)

        # ── Step 1: clear gradients ──────────────
        optimizer.zero_grad()

        # ── Step 2: forward pass ─────────────────
        output = model(src, trg, TEACHER_FORCE_RATIO)
        # output: (batch, trg_len, vocab_size)

        # ── Step 3: reshape for loss ─────────────
        output = output[:, 1:, :].reshape(-1, len(bn_vocab))
        trg    = trg[:, 1:].reshape(-1)
        #              ↑
        #         skip <SOS> token

        # ── Step 4: calculate loss ───────────────
        loss = criterion(output, trg)

        # ── Step 5: backward pass ────────────────
        loss.backward()

        # ── Step 6: clip gradients ───────────────
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        #                                                    ↑
        #                                       prevents exploding gradients

        # ── Step 7: update weights ───────────────
        optimizer.step()

        epoch_loss += loss.item()

    return epoch_loss / len(dataloader)    # average loss

In [58]:
def evaluate(model, dataloader, criterion):
    model.eval()                           # turn off dropout etc
    epoch_loss = 0

    with torch.no_grad():                  # no gradient calculation
        for src, trg in dataloader:
            if isinstance(src, list):
                src = torch.stack(src)     # Stack list of tensors into a single tensor
            src = src.to(device)

            if isinstance(trg, list):
                trg = torch.stack(trg)     # Stack list of tensors into a single tensor
            trg = trg.to(device)

            output = model(src, trg, teacher_forcing_ratio=0)
            #                                              ↑
            #                                no teacher forcing in eval

            output = output[:, 1:, :].reshape(-1, len(bn_vocab))
            trg    = trg[:, 1:].reshape(-1)

            loss = criterion(output, trg)
            epoch_loss += loss.item()

    return epoch_loss / len(dataloader)

In [63]:
best_loss = float('inf')

for epoch in range(EPOCHS):

    train_loss = train(model, data_loader, optimizer, criterion, CLIP)


    # save best model


    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"  Train Loss: {train_loss:.4f}")

Epoch 1/10
  Train Loss: 6.8295
Epoch 2/10
  Train Loss: 6.2829
Epoch 3/10
  Train Loss: 5.8860
Epoch 4/10
  Train Loss: 5.5947
Epoch 5/10
  Train Loss: 5.3048
Epoch 6/10
  Train Loss: 5.1303
Epoch 7/10
  Train Loss: 4.9153
Epoch 8/10
  Train Loss: 4.7462
Epoch 9/10
  Train Loss: 4.5895
Epoch 10/10
  Train Loss: 4.4653


In [64]:
def preprocess_english_sentence(sentence, en_vocab, en_max_len):
    # Tokenize and convert to lowercase
    tokenized_sentence = tokenizing(sentence)
    # Convert words to numerical IDs
    numerical_sentence = en_converter(tokenized_sentence)
    # Pad the sequence
    padded_sentence = padding(numerical_sentence, en_max_len)
    return torch.tensor(padded_sentence, dtype=torch.long).unsqueeze(0).to(device) # Add batch dimension and move to device

In [65]:
def predict_bangla_sentence(model, english_sentence, en_vocab, bn_vocab, en_max_len, bn_max_len, device):
    model.eval() # Set model to evaluation mode

    # Preprocess the English input
    input_tensor = preprocess_english_sentence(english_sentence, en_vocab, en_max_len)

    # Encode the input sentence
    encoder_output, hidden, cell = model.encoder(input_tensor)

    # Prepare the first input to the decoder (SOS token)
    bn_itos = {v: k for k, v in bn_vocab.items()} # Invert bn_vocab for index to string conversion
    trg_indexes = [bn_vocab['<sos>']]
    input_decoder = torch.tensor([bn_vocab['<sos>']], dtype=torch.long).to(device) # Initial input is <sos>

    for _ in range(bn_max_len - 1): # Iterate up to max output length
        # Decode one word at a time
        output, hidden, cell = model.decoder(input_decoder, hidden, cell, encoder_output)

        # Get the predicted next word
        pred_token = output.argmax(1).item()
        trg_indexes.append(pred_token)

        # If EOS token is predicted, stop decoding
        if pred_token == bn_vocab['<eos>']:
            break

        # Use the predicted word as the next input to the decoder
        input_decoder = torch.tensor([pred_token], dtype=torch.long).to(device)

    # Convert numerical IDs back to words
    predicted_sentence = [bn_itos[idx] for idx in trg_indexes]

    # Remove <sos> and <eos> tokens for final output
    if predicted_sentence[0] == '<sos>':
        predicted_sentence = predicted_sentence[1:]
    if '<eos>' in predicted_sentence:
        predicted_sentence = predicted_sentence[:predicted_sentence.index('<eos>')]

    return ' '.join(predicted_sentence)


In [66]:
# Example usage:
english_sentence = "A man is drinking water."
predicted_bn = predict_bangla_sentence(model, english_sentence, en_vocab, bn_vocab, en_max_len, bn_max_len, device)
print(f"English: {english_sentence}")
print(f"Bangla: {predicted_bn}")

English: A man is drinking water.
Bangla: একটি একজন লোক দেখছে থেকে দিকে থেকে দিকে হচ্ছে করছে করছে করছে করছে দিকে তাকিয়ে আছে দেখছে করছে করছে থেকে দিকে চেষ্টা করছে থেকে দিকে চেষ্টা করছে থেকে দিকে চেষ্টা করছে
